In [1]:
import sys
sys.path.append("..")

In [5]:
import tqdm
import os
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, theta_0, x_r, theta_r=None):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

In [7]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset):
    alpha = params['alpha']
    lamb = params['lamb']
    recourse_i = params['recourse_index']
    f_name = f'../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl'
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)

    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        x_r = recourse.get_recourse(x_0)
        append_result(results, recourse.name, seed, alpha, lamb, recourse_i[i], x_0, theta_0, x_r)

    df_results = pd.DataFrame(results)
    if params['append_results'] and os.path.exists(f_name):
        df_tmp = pd.read_pickle(f_name)
        df_results = pd.concat((df_tmp, df_results), axis=0).sort_values(['i'], ignore_index=True)

    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f_name)
    
    return df_results

In [ ]:
# def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
#     alpha = params['alpha']
#     lamb = params['lamb']
    
#     for seed in params['seeds']:
#         train_data, test_data = dataset.get_data(seed)
#         X_train, y_train = train_data
#         X_test, y_test = test_data
        
#         base_model = LR()
#         base_model.train(X_train.values, y_train.values)
        
#         weights_0 = base_model.model.coef_[0]
#         bias_0 = base_model.model.intercept_
        
#         recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
#         recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)

#         # <------------------------
#         # rng = np.random.default_rng(seed=seed)
#         # size_N = int(np.rint(0.15 * recourse_needed_X_test.shape[0]))
#         # recourse_needed_X_test = rng.choice(recourse_needed_X_test, size=size_N, replace=False) 
#         # <------------------------
        
#         for recourse_fn in recourse_fns:
#             recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha, lamb=lamb)
#             if params["lamb"] is None:
#                 params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
#                 recourse.lamb = params['lamb']
            
#             df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
#             results.append(df_results)

In [ ]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    lamb = params['lamb']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        for recourse_fn in recourse_fns:
            recourse_needed_X_test_idx = np.arange(recourse_needed_X_test.shape[0])

            recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha, lamb=lamb)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            if params['append_results']:
                f_name = f"../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl"
                if os.path.exists(f_name):
                    df_tmp = pd.read_pickle(f_name)
                    recourse_needed_X_test_idx = np.setdiff1d(recourse_needed_X_test_idx, df_tmp['i'].to_numpy())
                else:
                    print(f"{f_name} does not exist. The recourses will be created in a new file")

                if recourse_needed_X_test_idx.size == 0:
                    print(f"{f_name} already has all the recourses. Skipping")
                    continue
            
            if params['subsample']:
                rng = np.random.default_rng(seed=seed)
                size_N = int(np.rint(params['subsample_size'] * recourse_needed_X_test.shape[0]))
                if recourse_needed_X_test_idx.shape[0] < size_N:
                    size_N = recourse_needed_X_test_idx.shape[0]
                recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=size_N, replace=False)
            
            params['recourse_index'] = recourse_needed_X_test_idx.copy()
            recourse_needed_X_test = recourse_needed_X_test[recourse_needed_X_test_idx]
                
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
            results.append(df_results)

In [9]:
alphas = [0.1] # <------------------------
lambdas = [2.1] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:

        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True
        params['append_results'] = True
        params['subsample'] = True
        params['subsample_size'] = 0.075

        datasets = [SBADataset()] # <------------------------
        recourse_fns = [L1Recourse] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running sba data...


[L1PSD] [alpha=0.1] [lambda=2.1]:   0%|          | 0/3 [00:13<?, ?it/s]


KeyboardInterrupt: 

In [14]:
alphas = [0.02] # <------------------------
lambdas = [1.1, 0.1] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:

        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True

        datasets = [SBADataset()] # <------------------------
        recourse_fns = [ROARLInf] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running sba data...


[ROARLInf] [alpha=0.02] [lambda=1.1]: 100%|██████████| 39/39 [00:49<00:00,  1.28s/it]


[ROARLInf] Saving results for sba run 0


[ROARLInf] [alpha=0.02] [lambda=1.1]: 100%|██████████| 36/36 [00:22<00:00,  1.60it/s]


[ROARLInf] Saving results for sba run 1


...

[ROARLInf] Saving results for sba run 2


[ROARLInf] [alpha=0.02] [lambda=0.1]: 100%|██████████| 36/36 [03:22<00:00,  5.63s/it]


[ROARLInf] Saving results for sba run 3


[ROARLInf] [alpha=0.02] [lambda=0.1]: 100%|██████████| 38/38 [04:03<00:00,  6.42s/it]

[ROARLInf] Saving results for sba run 4
Finished sba

